In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/shah9212/notebook-ssg/__results__.html
/kaggle/input/notebooks/shah9212/notebook-ssg/__notebook__.ipynb
/kaggle/input/notebooks/shah9212/notebook-ssg/__output__.json
/kaggle/input/notebooks/shah9212/notebook-ssg/results.zip
/kaggle/input/notebooks/shah9212/notebook-ssg/custom.css
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/LICENSE
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/yolov8m.pt
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/.gitignore
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/README.md
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/requirements.txt
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/yolo26n.pt
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/setup.py
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/runs/detect/det/yolov8m_spatial/BoxR_curve.png
/kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/runs/detect/det/yolov8m_spatial

In [2]:
%%bash
git clone https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark && pip install -e . -q
pip install -q ultralytics hydra-core omegaconf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 11.4 MB/s eta 0:00:00


Cloning into 'SGG-Benchmark'...


In [3]:
%%bash
cd /kaggle/working/SGG-Benchmark
SRC=/kaggle/input/datasets/shah9212/spatial-sgg

echo "=== confirming contents ==="
ls -la "$SRC"

cp -r "$SRC/spatial_sgg" datasets/
mkdir -p configs/hydra/Spatial
cp "$SRC/spatial_sgg_react.yaml" configs/hydra/Spatial/react.yaml

for s in train val test; do
  test -f datasets/spatial_sgg/$s/_annotations.vlm.coco.json \
    || { echo "MISSING vlm variant in $s - re-export and re-upload"; exit 1; }
done
echo "vlm variant present in all three splits"

=== confirming contents ===
total 8
drwxr-xr-x 4 nobody nogroup    0 Aug  3 17:44 .
drwxr-xr-x 3 root   root    4096 Aug  3 23:13 ..
drwxr-xr-x 5 nobody nogroup    0 Aug  3 17:44 spatial_sgg
-rw-r--r-- 1 nobody nogroup 3703 Aug  3 17:44 spatial_sgg_react.yaml
drwxr-xr-x 4 nobody nogroup    0 Aug  3 17:44 spatial_sgg_yolo
vlm variant present in all three splits


In [4]:
import os, shutil, glob
os.chdir("/kaggle/working/SGG-Benchmark")
os.makedirs("checkpoints/BACKBONES", exist_ok=True)
src = glob.glob("/kaggle/input/**/yolov8m_spatial.pt", recursive=True)
assert src, "add the original notebook's committed output as an Input"
shutil.copy(src[0], "checkpoints/BACKBONES/yolov8m_spatial.pt")
print("frozen backbone from", src[0])
print("all three arms and all seeds share this detector, as in the first run")

frozen backbone from /kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/checkpoints/BACKBONES/yolov8m_spatial.pt
all three arms and all seeds share this detector, as in the first run


In [5]:
p = "/kaggle/working/SGG-Benchmark/sgg_benchmark/modeling/backbone/yolo.py"
src = open(p, encoding="utf-8").read()
old = "from ultralytics.utils.plotting import feature_visualization"
new = ("try:\n"
       "    from ultralytics.utils.plotting import feature_visualization\n"
       "except ImportError:\n"
       "    feature_visualization = None  # removed in newer ultralytics; only used\n"
       "    # by an optional debug-visualization path this run never enables")
assert old in src, "import line not found - the file may have changed further upstream"
src = src.replace(old, new, 1)
open(p, "w", encoding="utf-8").write(src)
print("patched:", p)

patched: /kaggle/working/SGG-Benchmark/sgg_benchmark/modeling/backbone/yolo.py


In [6]:
import pathlib
root = pathlib.Path("/kaggle/working/SGG-Benchmark/sgg_benchmark")
old = "from ultralytics.utils.plotting import feature_visualization"
marker = "feature_visualization = None  # removed in newer ultralytics"
new = ("try:\n"
       "    from ultralytics.utils.plotting import feature_visualization\n"
       "except ImportError:\n"
       "    feature_visualization = None  # removed in newer ultralytics; only used\n"
       "    # by an optional debug-visualization path this run never enables")

patched = []
for f in root.rglob("*.py"):
    src = f.read_text(encoding="utf-8")
    if marker in src:
        continue  # already patched — skip, do not touch again
    if old in src:
        f.write_text(src.replace(old, new, 1), encoding="utf-8")
        patched.append(str(f.relative_to(root)))

print(f"patched {len(patched)} file(s):")
for p in patched:
    print(" ", p)

patched 1 file(s):
  modeling/backbone/yoloe.py


In [7]:
%%bash
cd /kaggle/working/SGG-Benchmark
for seed in 42 43 44; do
  # train/val carry the VLM's relations; test is ALWAYS human gold, for
  # every arm, or the arms are not being scored against the same yardstick
  for s in train val; do
    cp datasets/spatial_sgg/$s/_annotations.vlm.coco.json \
       datasets/spatial_sgg/$s/_annotations.coco.json
  done
  cp datasets/spatial_sgg/test/_annotations.human.coco.json \
     datasets/spatial_sgg/test/_annotations.coco.json

  echo "=== react_vlm_s${seed} ==="
  python tools/relation_train_net_hydra.py \
    --config-path ../configs/hydra/Spatial --config-name react \
    --task sgdet --save-best seed=${seed} \
    output_dir=./checkpoints/spatial/react_vlm_s${seed}
done

=== react_vlm_s42 ===
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using Hydra + OmegaConf config mode...
Config path: ../configs/hydra/Spatial
Config name: react
Mode: TRAINING
Loading config from: /kaggle/working/SGG-Benchmark/configs/hydra/Spatial/react.yaml
2026-08-03 23:14:15,650 sgg_benchmark INFO: Using 1 GPUs
2026-08-03 23:14:15,650 sgg_benchmark INFO: Task mode: sgdet
2026-08-03 23:14:15,650 sgg_benchmark INFO: Loading training dataset to extract class information...
2026-08-03 23:14:15,753 sgg_benchmark INFO: Extracted from dataset: num_obj_classes=7, num_rel_classes=8
2026-08-03 23:14:15,754 sgg_benchmark INFO: Saving config to: ./checkpoints/spatial/react_vlm_s42/config.yml
2026-08-03 23:14:15,781 sgg_benchmark I

glove.6B.200d: 862MB [02:54, 4.93MB/s]                           
loading word vectors from ./datasets/glove.6B.200d.txt: 100%|██████████| 400000/400000 [00:20<00:00, 19542.30it/s]
/kaggle/working/SGG-Benchmark/sgg_benchmark/engine/trainer.py:334: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"{_ALIASES.get(k, k)}={float(v):.3f}" for k, v in loss_dict_reduced.items()
SGG Eval: 100%|██████████| 98/98 [00:00<00:00, 99.21it/s]
/kaggle/working/SGG-Benchmark/sgg_benchmark/engine/trainer.py:334: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"{_ALIASES.get(k, k)}={float(v):.3f}" for k, v in loss_dict_reduced.i

In [8]:
import os, torch, logging, glob, shutil
os.chdir("/kaggle/working/SGG-Benchmark")
from omegaconf import OmegaConf
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer
from sgg_benchmark.data import make_data_loader
from sgg_benchmark.engine.inference import inference

try:
    from sgg_benchmark.utils.logger import setup_logger
    logger = setup_logger("sgg_benchmark", ".", 0, verbose="INFO", steps=True)
except Exception:
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("sgg_benchmark")

for seed in [42, 43, 44]:
    tag = f"react_vlm_s{seed}"
    assert os.path.exists(f"checkpoints/spatial/{tag}/hydra_config.yaml"), tag
    ck = glob.glob(f"checkpoints/spatial/{tag}/best_model_epoch_*.pth")
    assert ck, f"no best_model_epoch_*.pth under checkpoints/spatial/{tag}/"
    print(f"{tag}: OK - {os.path.basename(sorted(ck)[-1])}")

shutil.copy("datasets/spatial_sgg/test/_annotations.human.coco.json",
            "datasets/spatial_sgg/test/_annotations.coco.json")

for seed in [42, 43, 44]:
    tag = f"react_vlm_s{seed}"
    cfg = OmegaConf.load(f"checkpoints/spatial/{tag}/hydra_config.yaml")
    out = f"./checkpoints/spatial/eval_{tag}"
    os.makedirs(out, exist_ok=True)
    cfg.output_dir = out
    ckpt = sorted(glob.glob(f"checkpoints/spatial/{tag}/best_model_epoch_*.pth"))[-1]

    model = build_detection_model(cfg).to(cfg.model.device)
    DetectronCheckpointer(cfg, model).load(ckpt)
    model.eval()
    loader = make_data_loader(cfg, mode="test")[0]

    print("=" * 80, flush=True)
    print(f"EVAL {tag} | {os.path.basename(ckpt)} | "
          f"test images: {len(loader.dataset)}", flush=True)
    print("=" * 80, flush=True)
    with torch.no_grad():
        inference(cfg, model, loader, dataset_name="SpatialRobot_test",
                  iou_types=("bbox", "relations"), box_only=False,
                  device=cfg.model.device, expected_results=[],
                  expected_results_sigma_tol=4, output_folder=out, logger=logger)
print("\nALL EVALUATIONS DONE")


react_vlm_s42: OK - best_model_epoch_20.pth
react_vlm_s43: OK - best_model_epoch_23.pth
react_vlm_s44: OK - best_model_epoch_14.pth
Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   32

100%|██████████| 210/210 [00:08<00:00, 24.06it/s]

2026-08-03 23:58:07,301 sgg_benchmark INFO: Total run time: 0:00:08 (39.020195861089796 ms / img per device, on 1 devices)
2026-08-03 23:58:07,303 sgg_benchmark INFO: Average latency per image: 39.020195861089796ms
2026-08-03 23:58:07,303 sgg_benchmark INFO: Standard deviation of latency: 53.70309742087589ms


2026-08-03 23:58:07,371 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-03 23:58:07,372 sgg_benchmark.data.build INFO: get dataset statistics...
2026-08-03 23:58:07,372 sgg_benchmark.data.build INFO: Loading data statistics from: ./checkpoints/spatial/eval_react_vlm_s42/SpatialRobot_statistics.cache
2026-08-03 23:58:07,373 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-03 23:58:07,377 sgg_benchmark INFO: Dynamically loaded 188 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(7390, 7)
0/7390
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.73s).
Accumulating evaluation results...
DONE (t=0.14s).
 Average Precision  (AP) @

SGG Eval: 100%|██████████| 210/210 [00:01<00:00, 173.94it/s]

...

100%|██████████| 210/210 [00:07<00:00, 26.53it/s]

2026-08-03 23:58:39,059 sgg_benchmark INFO: Total run time: 0:00:07 (34.872503008161274 ms / img per device, on 1 devices)
2026-08-03 23:58:39,060 sgg_benchmark INFO: Average latency per image: 34.872503008161274ms
2026-08-03 23:58:39,061 sgg_benchmark INFO: Standard deviation of latency: 3.2804770477326812ms


2026-08-03 23:58:39,136 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-03 23:58:39,136 sgg_benchmark.data.build INFO: get dataset statistics...
2026-08-03 23:58:39,137 sgg_benchmark.data.build INFO: Loading data statistics from: ./checkpoints/spatial/eval_react_vlm_s44/SpatialRobot_statistics.cache
2026-08-03 23:58:39,138 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-03 23:58:39,141 sgg_benchmark INFO: Dynamically loaded 188 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(7390, 7)
0/7390
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.03s).
Accumulating evaluation results...
DONE (t=0.14s).
 Average Precision  (AP) @

SGG Eval: 100%|██████████| 210/210 [00:01<00:00, 175.70it/s]

2026-08-03 23:58:41,678 sgg_benchmark INFO: 
Detection evaluation mAp=0.6541
SGG eval:     R @ 20: 0.1796;     R @ 50: 0.2490;     R @ 100: 0.2955;  for mode=sgdet.
SGG eval:    mR @ 20: 0.2010;    mR @ 50: 0.2664;    mR @ 100: 0.3142;  for mode=sgdet.
----------------------- Details ------------------------
(on:0.6402) (under:0.6063) (to the left of:0.3022) (to the right of:0.3770) (in front of:0.1041) (behind:0.0977) (near:0.0721) 
--------------------------------------------------------
SGG eval:     zR @ 20: 0.0000;     zR @ 50: 0.0000;     zR @ 100: 0.0000;  for mode=sgdet.
SGG eval:     F1 @ 20: 0.1897;     F1 @ 50: 0.2574;     F1 @ 100: 0.3046;  for mode=sgdet.


ALL EVALUATIONS DONE


In [9]:
%%bash
cd /kaggle/working/SGG-Benchmark
zip -rq /kaggle/working/vlm_results.zip checkpoints/spatial -x "*.pth" -x "*.pt"
echo "RESULTS -> /kaggle/working/vlm_results.zip"
ls -la /kaggle/working/vlm_results.zip

RESULTS -> /kaggle/working/vlm_results.zip
-rw-r--r-- 1 root root 111502 Aug  3 23:58 /kaggle/working/vlm_results.zip
